# Optimisation pour l'apprentissage automatique — M2 IASD 2025/2026

## Projet : Méthodes primales-duales et transfert de couleurs

Ce notebook présente notre travail sur les méthodes primales-duales appliquées au transport optimal, dans le cadre du projet du cours OAA (C. W. Royer, Dauphine–PSL).

Le fil directeur est le suivant : on part de méthodes d'optimisation générales (GDA, PDHG) testées d'abord sur des problèmes jouets simples, puis appliquées à une tâche concrète de traitement d'image — le **transfert de couleurs** entre deux photographies.

Le notebook est organisé en trois parties :

- **Partie 1** (Q1 à Q3) : méthodes primales-duales de base sur un problème linéaire contraint
- **Partie 2** (Q4 à Q6) : transport optimal et transfert de couleurs, comparaison avec Sinkhorn
- **Partie 3** (Q7 et Q8) : version déséquilibrée (relaxation quadratique), méthodes PGD et NAPG

## Installation et imports

In [ ]:
!pip install POT -q

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import linprog
import time
import os

plt.rcParams.update({"font.size": 11, "figure.dpi": 110})
print("Imports OK.")

---

# Partie 1 — Méthodes primales-duales de base

## Contexte

On considère le programme linéaire :

$$\min_{w \geq 0} c^\top w \quad \text{s.c.} \quad Aw = b$$

avec $A \in \mathbb{R}^{r \times d}$, $b \in \mathbb{R}^r$, $c \in \mathbb{R}^d$. On relâche la contrainte d'égalité via un multiplicateur de Lagrange $z \in \mathbb{R}^r$ pour former le Lagrangien :

$$\mathcal{L}(w, z) = c^\top w - z^\top (Aw - b)$$

On cherche un **point-selle** $(w^*, z^*)$ vérifiant $\min_{w \geq 0} \max_z \mathcal{L}(w,z)$. Les méthodes présentées ci-dessous optimisent simultanément sur $w$ (descente) et sur $z$ (montée).

In [ ]:
def proj_pos(v):
    """Projection euclidienne sur l'orthant positif : max(0, v) composante par composante."""
    return np.maximum(0.0, v)

---

## Question 1 — Gradient Descent-Ascent (GDA)

GDA applique simultanément une descente de gradient sur $w$ et une montée de gradient sur $z$, en utilisant la valeur courante de l'autre variable. Les gradients partiels du Lagrangien sont :

$$\nabla_w \mathcal{L}(w,z) = c - A^\top z, \qquad \nabla_z \mathcal{L}(w,z) = -(Aw - b)$$

L'itération s'écrit :
$$w_{k+1} = \mathrm{proj}_{\geq 0}\bigl[w_k - \alpha_k(c - A^\top z_k)\bigr], \qquad z_{k+1} = z_k - \beta_k(Aw_k - b)$$

La mise à jour de $z$ utilise $w_k$ avant la mise à jour primale — c'est la forme simultanée de GDA.

In [ ]:
def gda(c, A, b, w0, z0, alpha, beta, n_iters=500):
    w = w0.copy().astype(float)
    z = z0.copy().astype(float)
    ws, zs = [w.copy()], [z.copy()]
    for _ in range(n_iters):
        grad_w = c - A.T @ z
        w_new = proj_pos(w - alpha * grad_w)
        z_new = z - beta * (A @ w - b)  # utilise w_k avant mise a jour
        w, z = w_new, z_new
        ws.append(w.copy())
        zs.append(z.copy())
    return np.array(ws), np.array(zs)

### Application au problème jouet

On teste GDA sur le problème scalaire $\min_{w \geq 0} \max_{z} (w-3)z$, qui correspond au LP avec $c = 0$, $A = [-1]$, $b = [-3]$. La solution exacte est $w^* = 3$, $z^* = 0$ (pour $w \neq 3$, le supremum sur $z$ est infini, donc le minimum est atteint en $w = 3$).

In [ ]:
c_toy = np.array([0.0])
A_toy = np.array([[-1.0]])
b_toy = np.array([-3.0])
w_star = np.array([3.0])
z_star = np.array([0.0])
w0 = np.array([2.0])
z0 = np.array([2.0])
ALPHA = BETA = 0.2
N = 300

ws_gda, zs_gda = gda(c_toy, A_toy, b_toy, w0, z0, ALPHA, BETA, N)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
iters = np.arange(N + 1)

axes[0].plot(iters, ws_gda[:, 0], color="tab:red", lw=1.5, label="GDA")
axes[0].axhline(3.0, color="k", ls=":", lw=1, label="$w^*=3$")
axes[0].set(title="Trajectoire de $w_k$ (GDA)", xlabel="Itération $k$", ylabel="$w_k$")
axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(iters, zs_gda[:, 0], color="tab:red", lw=1.5, label="GDA")
axes[1].axhline(0.0, color="k", ls=":", lw=1, label="$z^*=0$")
axes[1].set(title="Trajectoire de $z_k$ (GDA)", xlabel="Itération $k$", ylabel="$z_k$")
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.suptitle(r"Q1 — GDA : $\min_{w \geq 0} \max_z (w-3)z$, $\alpha=\beta=0.2$", fontweight="bold")
plt.tight_layout()
plt.show()

### Observation (Q1)

On observe que $w_k$ et $z_k$ **oscillent avec une amplitude qui reste bornée mais ne diminue pas** : GDA ne converge pas vers le point-selle $(3, 0)$. Les itérés tournent autour de la solution sans s'en approcher.

Ce comportement s'explique par l'analyse spectrale du jacobien de GDA au voisinage du point-selle. En linéarisant l'itération, le jacobien est :

$$J_{\text{GDA}} = \begin{pmatrix} 1 & -\alpha \\ \beta & 1 \end{pmatrix}$$

dont les valeurs propres vérifient $|\lambda| = \sqrt{1 + \alpha\beta} = \sqrt{1.04} \approx 1.02 > 1$ pour $\alpha = \beta = 0.2$. Un rayon spectral supérieur à 1 implique une divergence. En pratique, la projection $\mathrm{proj}_{\geq 0}$ borne les valeurs de $w_k$ par le bas, ce qui explique que les oscillations semblent d'amplitude constante plutôt qu'explosives — mais GDA ne converge pas pour autant.

---

## Question 2 — GDA à pas alternants (Alt-GDA)

Une variante récente propose d'alterner les signes des pas selon la parité de l'itération :

$$\alpha_{2k} = -\beta_{2k} = -\alpha_{2k+1} = \beta_{2k+1} = 0.2$$

Concrètement :
- Itération paire : $\alpha_k = +0.2$ (descente sur $w$), $\beta_k = -0.2$ (descente sur $z$ aussi)
- Itération impaire : $\alpha_k = -0.2$ (montée sur $w$), $\beta_k = +0.2$ (montée sur $z$)

Aux itérations impaires, on effectue une **montée** de gradient sur $w$, ce qui va à l'encontre de l'objectif de minimisation. C'est ce qui rend cette proposition surprenante : comment une méthode qui, un pas sur deux, fait l'opposé de ce qu'on veut peut-elle converger ?

In [ ]:
def alt_gda(c, A, b, w0, z0, step=0.2, n_iters=500):
    w = w0.copy().astype(float)
    z = z0.copy().astype(float)
    ws, zs = [w.copy()], [z.copy()]
    for k in range(n_iters):
        if k % 2 == 0:
            alpha_k, beta_k = +step, -step
        else:
            alpha_k, beta_k = -step, +step
        grad_w = c - A.T @ z
        w_new = proj_pos(w - alpha_k * grad_w)
        z_new = z - beta_k * (A @ w - b)
        w, z = w_new, z_new
        ws.append(w.copy())
        zs.append(z.copy())
    return np.array(ws), np.array(zs)

### Pourquoi Alt-GDA converge-t-il malgré tout ?

La clé est de regarder ce qui se passe sur **deux itérations consécutives** plutôt que sur chaque pas individuellement. En posant $u_k = w_k - w^*$ et en développant :

- Après l'itération paire : $u_{k+1} = u_k - s \cdot z_k$ et $z_{k+1} = z_k - s \cdot u_k$
- Après l'itération impaire : $u_{k+2} = (1 - s^2) u_k$ et $z_{k+2} = (1 - s^2) z_k$

Après deux pas, les deux variables sont contractées d'un facteur $(1 - s^2) = 1 - 0.04 = 0.96 < 1$. La convergence est donc linéaire avec taux $O(0.96^{k/2})$ par itération. Les pas négatifs ne nuisent pas : ils préparent en réalité la contraction du pas suivant.

In [ ]:
ws_alt, zs_alt = alt_gda(c_toy, A_toy, b_toy, w0, z0, step=0.2, n_iters=N)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
dist_gda = np.linalg.norm(ws_gda - w_star, axis=1)
dist_alt = np.linalg.norm(ws_alt - w_star, axis=1)
rate = 0.96 ** (iters / 2)

axes[0].semilogy(iters, dist_gda + 1e-16, color="tab:red",    label="GDA",     lw=1.8, ls="--")
axes[0].semilogy(iters, dist_alt + 1e-16, color="tab:orange", label="Alt-GDA", lw=1.8)
axes[0].semilogy(iters, rate * dist_alt[0], color="gray",     label="Taux $0.96^{k/2}$", lw=1, ls=":")
axes[0].set(title=r"$\|w_k - w^*\|$ (log)", xlabel="Itération $k$")
axes[0].legend(); axes[0].grid(True, which="both", alpha=0.3)

axes[1].plot(ws_gda[:, 0], zs_gda[:, 0], color="tab:red",    lw=0.7, label="GDA")
axes[1].plot(ws_alt[:, 0], zs_alt[:, 0], color="tab:orange", lw=0.7, label="Alt-GDA")
axes[1].scatter([3.0], [0.0], s=200, marker="*", color="k", zorder=5, label="$(w^*, z^*)$")
axes[1].set(title="Portrait de phase $(w_k, z_k)$", xlabel="$w_k$", ylabel="$z_k$")
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.suptitle("Q2 — GDA diverge, Alt-GDA converge", fontweight="bold")
plt.tight_layout()
plt.show()

### Observation (Q2)

La courbe de convergence confirme la théorie : Alt-GDA suit une droite sur l'échelle logarithmique, caractéristique d'une convergence linéaire, avec une pente cohérente avec le taux $0.96^{k/2}$. Le portrait de phase montre clairement la différence : GDA tourne en orbite stable autour du point-selle sans s'en approcher, tandis qu'Alt-GDA converge en spirale resserrée vers $(3, 0)$.

---

## Question 3 — PDHG (Primal-Dual Hybrid Gradient)

PDHG (Chambolle-Pock, 2011) introduit un **point extrapolé** $\tilde{w}_{k+1} = 2w_{k+1} - w_k$ dans la mise à jour duale :

$$w_{k+1} = \mathrm{proj}_{\geq 0}\bigl[w_k - \alpha(c - A^\top z_k)\bigr]$$
$$z_{k+1} = z_k - \beta\bigl(A(2w_{k+1} - w_k) - b\bigr)$$

L'intuition est simple : au lieu d'utiliser $w_{k+1}$ ou $w_k$ dans la mise à jour duale, on utilise un point qui prolonge le déplacement primal $w_{k+1} - w_k$. Cette anticipation compense l'effet oscillant de GDA.

La condition suffisante de convergence est $\alpha \beta \|A\|^2 < 1$, soit $0.04 < 1$ ici. Le rayon spectral du jacobien de PDHG vaut $\sqrt{\det(J)} = \sqrt{0.96} \approx 0.98 < 1$, contre $\sqrt{1.04} > 1$ pour GDA.

In [ ]:
def pdhg(c, A, b, w0, z0, alpha, beta, n_iters=500):
    w = w0.copy().astype(float)
    z = z0.copy().astype(float)
    ws, zs = [w.copy()], [z.copy()]
    for _ in range(n_iters):
        grad_w = c - A.T @ z
        w_new = proj_pos(w - alpha * grad_w)
        w_extrap = 2.0 * w_new - w   # point extrapole : 2*w_{k+1} - w_k
        z_new = z - beta * (A @ w_extrap - b)
        w, z = w_new, z_new
        ws.append(w.copy())
        zs.append(z.copy())
    return np.array(ws), np.array(zs)

In [ ]:
ws_pdhg, zs_pdhg = pdhg(c_toy, A_toy, b_toy, w0, z0, ALPHA, BETA, N)
dist_pdhg = np.linalg.norm(ws_pdhg - w_star, axis=1)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

for ws, name, col, ls in [(ws_gda,  "GDA",     "tab:red",    "--"),
                            (ws_alt,  "Alt-GDA", "tab:orange", "-."),
                            (ws_pdhg, "PDHG",    "tab:blue",   "-")]:
    axes[0].plot(iters, ws[:, 0], label=name, color=col, ls=ls, lw=1.8)
axes[0].axhline(3.0, color="k", ls=":", lw=1)
axes[0].set(title="Trajectoire $w_k$", xlabel="Itération $k$")
axes[0].legend(); axes[0].grid(True, alpha=0.3)

for dist, name, col, ls in [(dist_gda,  "GDA",     "tab:red",    "--"),
                              (dist_alt,  "Alt-GDA", "tab:orange", "-."),
                              (dist_pdhg, "PDHG",    "tab:blue",   "-")]:
    axes[1].semilogy(iters, dist + 1e-16, label=name, color=col, ls=ls, lw=1.8)
axes[1].semilogy(iters, 0.96 ** (iters / 2) * dist_alt[0], color="gray", ls=":", lw=1, label="$0.96^{k/2}$")
axes[1].set(title=r"$\|w_k - w^*\|$ (log)", xlabel="Itération $k$")
axes[1].legend(); axes[1].grid(True, which="both", alpha=0.3)

axes[2].plot(ws_gda[:, 0],  zs_gda[:, 0],  color="tab:red",  lw=0.7, label="GDA")
axes[2].plot(ws_pdhg[:, 0], zs_pdhg[:, 0], color="tab:blue", lw=0.7, label="PDHG")
axes[2].scatter([3.0], [0.0], s=200, marker="*", color="k", zorder=5)
axes[2].set(title="Portrait de phase", xlabel="$w_k$", ylabel="$z_k$")
axes[2].legend(); axes[2].grid(True, alpha=0.3)

plt.suptitle("Q3 — Comparaison GDA / Alt-GDA / PDHG", fontweight="bold")
plt.tight_layout()
plt.show()

print(f"Valeurs finales a k={N} :")
print(f"  GDA     : w = {ws_gda[-1,0]:.4f},  distance a w* = {dist_gda[-1]:.2e}")
print(f"  Alt-GDA : w = {ws_alt[-1,0]:.4f},  distance a w* = {dist_alt[-1]:.2e}")
print(f"  PDHG    : w = {ws_pdhg[-1,0]:.4f},  distance a w* = {dist_pdhg[-1]:.2e}")

### Observation (Q3)

PDHG converge vers $w^* = 3$ plus rapidement qu'Alt-GDA sur ce problème. Sur le portrait de phase, on voit nettement la différence de comportement : GDA tourne en orbite ouverte, tandis que PDHG converge en spirale vers le point-selle, avec un rayon qui décroît à chaque tour.

L'extrapolation $\tilde{w} = 2w_{k+1} - w_k$ dans la mise à jour duale est le mécanisme clé : en anticipant le prochain déplacement primal, PDHG évite l'accumulation de phase qui fait diverger GDA. Alt-GDA et PDHG atteignent tous deux un rayon spectral inférieur à 1 — $\sqrt{0.96}$ pour PDHG — mais par des mécanismes très différents.

**Bilan de la Partie 1 :** GDA échoue sur ce problème bilinéaire, quelle que soit la valeur du pas. Alt-GDA converge grâce à une annulation remarquable sur deux itérations. PDHG converge grâce à l'extrapolation primale, qui est la modification la plus naturelle et la plus généralisable des deux.

---

# Partie 2 — Transport optimal et transfert de couleurs

## Contexte

Le **transport optimal** consiste, étant données deux distributions $a, b \in \mathbb{R}^d$ avec $a^\top \mathbf{1}_d = b^\top \mathbf{1}_d = 1$, à trouver un plan de transport $P \in \mathbb{R}^{d \times d}$ minimisant le coût total :

$$\min_{P \geq 0} \langle C, P \rangle \quad \text{s.c.} \quad P\mathbf{1}_d = a, \quad P^\top\mathbf{1}_d = b$$

Ce problème est exactement un LP de la forme de la Partie 1 avec $w = \mathrm{vec}(P) \in \mathbb{R}^{d^2}$ et une matrice $A \in \mathbb{R}^{2d \times d^2}$ encodant les $2d$ contraintes de marge. On peut donc appliquer GDA et PDHG directement.

## Opérateurs $A$ et $A^\top$ sans matrice explicite

Former $A$ explicitement pour $d = 300$ donnerait une matrice $600 \times 90000 \approx 432$ Mo. On utilise plutôt les opérateurs implicites :

$$Aw = \begin{pmatrix} P\mathbf{1}_d \\ P^\top\mathbf{1}_d \end{pmatrix} \qquad \text{et} \qquad [A^\top(p, q)]_{ij} = p_i + q_j$$

Ces opérateurs coûtent $O(d^2)$ au lieu de $O(d^4)$ et sont mathématiquement équivalents. La norme spectrale de $A$ vaut $\|A\| = \sqrt{2d}$, ce qui donne la condition de pas pour PDHG : $\alpha \beta < 1/(2d)$.

In [ ]:
def apply_A(P):
    return np.concatenate([P.sum(axis=1), P.sum(axis=0)])

def apply_AT(p, q):
    return p[:, None] + q[None, :]

def build_A_dense(d):
    A = np.zeros((2 * d, d * d))
    for i in range(d):
        for j in range(d):
            col = i * d + j
            A[i, col] = 1.0
            A[d + j, col] = 1.0
    return A

def ot_metrics(C, a, b, P):
    return {
        "cost": float(np.sum(C * P)),
        "row_violation": float(np.linalg.norm(P.sum(axis=1) - a)),
        "col_violation": float(np.linalg.norm(P.sum(axis=0) - b)),
    }

def ot_gda(C, a, b, P0, p0, q0, alpha, beta, n_iters, track_every=1):
    P, p, q = P0.copy().astype(float), p0.copy().astype(float), q0.copy().astype(float)
    rhs = np.concatenate([a, b])
    history = {"iter": [], "cost": [], "row_violation": [], "col_violation": []}
    def _record(k):
        m = ot_metrics(C, a, b, P)
        history["iter"].append(k); history["cost"].append(m["cost"])
        history["row_violation"].append(m["row_violation"])
        history["col_violation"].append(m["col_violation"])
    _record(0)
    for k in range(1, n_iters + 1):
        grad_P = C - apply_AT(p, q)
        P_new = proj_pos(P - alpha * grad_P)
        z = np.concatenate([p, q]) - beta * (apply_A(P) - rhs)
        P, p, q = P_new, z[:len(a)], z[len(a):]
        if k % track_every == 0 or k == n_iters:
            _record(k)
    return P, p, q, history

def ot_pdhg(C, a, b, P0, p0, q0, alpha, beta, n_iters, track_every=1):
    P, p, q = P0.copy().astype(float), p0.copy().astype(float), q0.copy().astype(float)
    rhs = np.concatenate([a, b])
    history = {"iter": [], "cost": [], "row_violation": [], "col_violation": []}
    def _record(k):
        m = ot_metrics(C, a, b, P)
        history["iter"].append(k); history["cost"].append(m["cost"])
        history["row_violation"].append(m["row_violation"])
        history["col_violation"].append(m["col_violation"])
    _record(0)
    for k in range(1, n_iters + 1):
        grad_P = C - apply_AT(p, q)
        P_new = proj_pos(P - alpha * grad_P)
        P_extrap = 2.0 * P_new - P
        z = np.concatenate([p, q]) - beta * (apply_A(P_extrap) - rhs)
        P, p, q = P_new, z[:len(a)], z[len(a):]
        if k % track_every == 0 or k == n_iters:
            _record(k)
    return P, p, q, history

---

## Question 4 — Problème de transport jouet

On applique GDA et PDHG au problème jouet suivant ($d = 2$) :

$$a = \begin{pmatrix} 0.4 \\ 0.6 \end{pmatrix}, \quad b = \begin{pmatrix} 0.5 \\ 0.5 \end{pmatrix}, \quad C = \begin{pmatrix} 1 & 5 \\ 1 & 2 \end{pmatrix}$$

La solution exacte est calculée par `scipy.linprog` et sert de référence.

In [ ]:
a4 = np.array([0.4, 0.6])
b4 = np.array([0.5, 0.5])
C4 = np.array([[1.0, 5.0], [1.0, 2.0]])
d4 = 2

A4 = build_A_dense(d4)
res4 = linprog(C4.flatten(), A_eq=A4, b_eq=np.concatenate([a4, b4]),
               bounds=(0, None), method="highs")
P4_star = res4.x.reshape(d4, d4)
cost4_star = res4.fun

print(f"Solution exacte (linprog) :")
print(f"  P* = {P4_star}")
print(f"  cout optimal = {cost4_star:.6f}")
print(f"  Borne de pas PDHG : alpha < 1/sqrt(2*d) = {1/np.sqrt(2*d4):.4f}")

In [ ]:
N4 = 2000
P0 = np.zeros((d4, d4))
p0 = q0 = np.zeros(d4)

P_gda4,  _, _, hist_gda4  = ot_gda (C4, a4, b4, P0, p0, q0, 0.2, 0.2, N4)
P_pdhg4, _, _, hist_pdhg4 = ot_pdhg(C4, a4, b4, P0, p0, q0, 0.2, 0.2, N4)

m_gda4  = ot_metrics(C4, a4, b4, P_gda4)
m_pdhg4 = ot_metrics(C4, a4, b4, P_pdhg4)

print(f"Apres N={N4} iterations (alpha=beta=0.2) :")
print(f"  {'':10s}  {'cout':>8s}  {'row_viol':>10s}  {'col_viol':>10s}  {'dist P*':>10s}")
for name, m, P_m in [("GDA",   m_gda4,  P_gda4),
                      ("PDHG",  m_pdhg4, P_pdhg4),
                      ("Exact", {"cost": cost4_star, "row_violation": 0, "col_violation": 0}, P4_star)]:
    err = np.linalg.norm(P_m - P4_star)
    print(f"  {name:10s}  {m['cost']:>8.4f}  {m['row_violation']:>10.2e}  {m['col_violation']:>10.2e}  {err:>10.4f}")

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle("Q4 — GDA vs PDHG : transport optimal jouet", fontweight="bold")

axes[0].semilogy(hist_gda4["iter"],
                  np.abs(np.array(hist_gda4["cost"]) - cost4_star) + 1e-16,
                  color="tab:red", label="GDA")
axes[0].semilogy(hist_pdhg4["iter"],
                  np.abs(np.array(hist_pdhg4["cost"]) - cost4_star) + 1e-16,
                  color="tab:blue", label="PDHG")
axes[0].set(title=r"$|\langle C, P_k\rangle - \mathrm{OPT}|$", xlabel="Itération $k$")
axes[0].legend(); axes[0].grid(True, which="both", alpha=0.3)

viol_gda4  = np.array(hist_gda4["row_violation"]) + np.array(hist_gda4["col_violation"])
viol_pdhg4 = np.array(hist_pdhg4["row_violation"]) + np.array(hist_pdhg4["col_violation"])
axes[1].semilogy(hist_gda4["iter"],  viol_gda4  + 1e-16, color="tab:red",  label="GDA")
axes[1].semilogy(hist_pdhg4["iter"], viol_pdhg4 + 1e-16, color="tab:blue", label="PDHG")
axes[1].set(title=r"Violation $\|P_k\mathbf{1}-a\|+\|P_k^\top\mathbf{1}-b\|$",
             xlabel="Itération $k$")
axes[1].legend(); axes[1].grid(True, which="both", alpha=0.3)

plt.tight_layout()
plt.show()

### Observation (Q4)

Les résultats numériques sont sans ambiguïté : après 2000 itérations, PDHG atteint $\|P - P^*\| \approx 0$ avec des violations de contraintes de l'ordre de $10^{-12}$ (essentiellement la précision machine), et un coût de 1.5000 identique à l'optimal. GDA, en revanche, reste bloqué avec des violations de l'ordre de 0.7 et un plan $P$ très éloigné de $P^*$.

Ce résultat est cohérent avec la Partie 1 : la structure mathématique du Lagrangien est identique (bilinéaire en $(P, z)$), et le comportement divergent de GDA sur le problème scalaire se retrouve en dimension $d^2 = 4$. La condition de pas de PDHG ($\alpha \beta \cdot 2d = 0.08 < 1$) est vérifiée, ce qui garantit la convergence.

---

## Questions 5 et 6 — Transfert de couleurs

On formule le transfert de couleurs comme un problème de transport optimal. Étant données deux images source $I_s$ et cible $I_t$, on sous-échantillonne $d$ pixels dans chacune :

- $X_s, X_t \in [0,1]^{d \times 3}$ : pixels source et cible (couleurs RGB normalisées)
- $a = b = \mathbf{1}_d$ : masses uniformes (chaque pixel a le même poids)
- $C_{ij} = \|[X_s]_i - [X_t]_j\|$ : distance euclidienne entre les couleurs

Le plan de transport $P^*$ encode comment redistribuer chaque couleur source vers les couleurs cibles. La couleur recolorisée de chaque pixel source est le barycentre pondéré par $P$ des couleurs cibles. L'image complète est ensuite recolorisée par recherche de plus proche voisin.

In [ ]:
def load_or_synthesize_images(src_path="ossau.jpg", tgt_path="rer.jpg",
                               fallback_shape=(64, 64), seed=0):
    if os.path.exists(src_path) and os.path.exists(tgt_path):
        from PIL import Image
        src = np.array(Image.open(src_path).convert("RGB")) / 255.0
        tgt = np.array(Image.open(tgt_path).convert("RGB")) / 255.0
        return src, tgt, False
    print(f"Images introuvables : generation d'images synthetiques.")
    rng = np.random.default_rng(seed)
    H, W = fallback_shape
    xx, yy = np.meshgrid(np.linspace(0, 1, W), np.linspace(0, 1, H))
    src = np.clip(np.stack([0.2 + 0.3*yy, 0.4 + 0.4*(1-yy), 0.2 + 0.2*xx], axis=-1)
                  + 0.05 * rng.standard_normal((H, W, 3)), 0, 1)
    tgt = np.clip(np.stack([0.3 + 0.5*yy, 0.3 + 0.2*yy, 0.5 + 0.4*(1-yy)], axis=-1)
                  + 0.05 * rng.standard_normal((H, W, 3)), 0, 1)
    return src, tgt, True

def sample_pixels(image, n_samples, seed=0):
    flat = image.reshape(-1, 3)
    idx = np.random.default_rng(seed).choice(len(flat), n_samples, replace=False)
    return flat[idx], idx

def transport_to_colors(P, a, Xt):
    return (P @ Xt) / (P.sum(axis=1, keepdims=True) + 1e-9)

def recolor_image(image, Xs, T_Xs):
    from sklearn.neighbors import NearestNeighbors
    flat = image.reshape(-1, 3)
    nn = NearestNeighbors(n_neighbors=1).fit(Xs)
    _, indices = nn.kneighbors(flat)
    out = np.clip(T_Xs[indices[:, 0]].reshape(image.shape), 0, 1)
    return (out * 255).astype(np.uint8)

def color_transfer_lp(src, tgt, method, n_samples=300, alpha=None, beta=None,
                       n_iters=4000, seed=0):
    d = n_samples
    Xs, _ = sample_pixels(src, d, seed=seed)
    Xt, _ = sample_pixels(tgt, d, seed=seed + 1)
    a = np.ones(d)
    b = np.ones(d)
    C = np.sqrt(((Xs[:, None, :] - Xt[None, :, :]) ** 2).sum(axis=-1))
    if alpha is None:
        alpha = 0.9 / np.sqrt(2 * d)
    if beta is None:
        beta = 0.9 / np.sqrt(2 * d)
    t0 = time.time()
    history = None
    if method == "sinkhorn":
        import ot as pot
        P = pot.sinkhorn(a, b, C, reg=1e-2, numItermax=n_iters)
    elif method == "gda":
        P, _, _, history = ot_gda(C, a, b, np.zeros((d, d)),
                                   np.zeros(d), np.zeros(d),
                                   alpha, beta, n_iters,
                                   track_every=max(1, n_iters // 200))
    elif method == "pdhg":
        P, _, _, history = ot_pdhg(C, a, b, np.zeros((d, d)),
                                    np.zeros(d), np.zeros(d),
                                    alpha, beta, n_iters,
                                    track_every=max(1, n_iters // 200))
    else:
        raise ValueError(f"Methode inconnue : {method!r}")
    runtime = time.time() - t0
    T_Xs = transport_to_colors(P, a, Xt)
    out = recolor_image(src, Xs, T_Xs)
    return {"out": out, "P": P, "metrics": ot_metrics(C, a, b, P),
            "history": history, "runtime": runtime, "Xs": Xs, "Xt": Xt, "C": C, "a": a, "b": b}

In [ ]:
src, tgt, is_synth = load_or_synthesize_images("ossau.jpg", "rer.jpg")
print(f"Images chargees : src {src.shape}, tgt {tgt.shape} "
      f"({'synthetiques' if is_synth else 'fichiers reels'})")

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].imshow(src); axes[0].set_title("Source"); axes[0].axis("off")
axes[1].imshow(tgt); axes[1].set_title("Cible"); axes[1].axis("off")
plt.suptitle("Images d'entrée", fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
N_SAMPLES = 300
N_ITERS_CT = 4000
alpha_ct = beta_ct = 0.9 / np.sqrt(2 * N_SAMPLES)

print(f"d = {N_SAMPLES} pixels echantillonnes")
print(f"alpha = beta = 0.9/sqrt(2d) = {alpha_ct:.5f}  (borne PDHG : {1/np.sqrt(2*N_SAMPLES):.5f})")
print(f"n_iters = {N_ITERS_CT}")
print()
print("Execution des methodes (peut prendre quelques minutes)...")

results_ct = {}
for method in ["sinkhorn", "gda", "pdhg"]:
    print(f"  {method.upper()}...", end=" ", flush=True)
    results_ct[method] = color_transfer_lp(src, tgt, method=method,
                                            n_samples=N_SAMPLES,
                                            alpha=alpha_ct, beta=beta_ct,
                                            n_iters=N_ITERS_CT, seed=0)
    m = results_ct[method]["metrics"]
    print(f"cout={m['cost']:.4f}  row_viol={m['row_violation']:.2e}  "
          f"col_viol={m['col_violation']:.2e}  t={results_ct[method]['runtime']:.1f}s")

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(20, 4))
fig.suptitle("Q5-Q6 — Transfert de couleurs : comparaison des méthodes", fontweight="bold")

axes[0].imshow(src); axes[0].set_title("Source"); axes[0].axis("off")
axes[1].imshow(tgt); axes[1].set_title("Cible"); axes[1].axis("off")
for ax, method in zip(axes[2:], ["sinkhorn", "gda", "pdhg"]):
    ax.imshow(results_ct[method]["out"])
    ax.set_title(method.upper())
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle("Q5-Q6 — Convergence GDA vs PDHG (transfert de couleurs)", fontweight="bold")

for method, col in [("gda", "tab:red"), ("pdhg", "tab:blue")]:
    h = results_ct[method]["history"]
    axes[0].plot(h["iter"], h["cost"], color=col, label=method.upper(), lw=1.5)
    viol = np.array(h["row_violation"]) + np.array(h["col_violation"])
    axes[1].semilogy(h["iter"], viol + 1e-16, color=col, label=method.upper(), lw=1.5)

axes[0].axhline(results_ct["sinkhorn"]["metrics"]["cost"], color="gray",
                ls=":", label="Sinkhorn (ref.)")
axes[0].set(title=r"Coût $\langle C, P_k\rangle$", xlabel="Itération $k$")
axes[1].set(title=r"Violation $\|P_k\mathbf{1}-a\|+\|P_k^\top\mathbf{1}-b\|$",
             xlabel="Itération $k$")
for ax in axes:
    ax.legend(); ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### Observation (Q6)

Les résultats obtenus illustrent parfaitement les différences entre les trois approches.

**GDA** produit une image corrompue (pixels noirs), ce qui est directement imputable à sa non-convergence : le plan $P_k$ oscille sans se stabiliser, les sommes de lignes de $P_k$ sont très loin de $a$ (violation de l'ordre de $10^1$ sur le graphe de droite), ce qui rend le barycentrage `transport_to_colors` numériquement instable et génère des valeurs de couleur hors de $[0,1]$. Ce résultat visuel confirme ce qu'on observait déjà numériquement en Q4.

**PDHG** produit une image de bonne qualité, avec des violations de contraintes qui descendent en dessous de $10^{-2}$ et un coût qui se stabilise. La convergence est lente comparée à Sinkhorn, mais le plan obtenu est une vraie solution du LP non régularisé.

**Sinkhorn** est clairement plus rapide et donne un résultat visuellement propre. Il faut cependant noter qu'il ne résout **pas** le même problème : la régularisation entropique avec $\varepsilon = 10^{-2}$ modifie l'objectif, ce qui donne un plan de transport dense (toutes les entrées de $P$ sont strictement positives) plutôt que parcimonieux comme le LP exact. Le coût obtenu par Sinkhorn est légèrement différent de celui de PDHG pour cette raison.

---

# Partie 3 — Extension : transport optimal déséquilibré

## Contexte

Dans les parties précédentes, les contraintes de marge $P\mathbf{1}_d = a$ et $P^\top\mathbf{1}_d = b$ sont imposées exactement via des multiplicateurs de Lagrange. Une alternative est de les **relaxer** en pénalités quadratiques :

$$\min_{P \geq 0} f(P) := \langle C, P\rangle + \frac{\lambda}{2}\|P\mathbf{1}_d - a\|^2 + \frac{\lambda}{2}\|P^\top\mathbf{1}_d - b\|^2$$

Pour $\lambda \to +\infty$, la solution converge vers celle du LP contraint. L'avantage est que $f$ est maintenant une fonction **lisse** (gradient $L$-Lipschitz avec $L = 2\lambda d$), ce qui permet d'utiliser des méthodes de gradient pures sans variable duale.

Le gradient s'écrit, en posant $r = P\mathbf{1}_d - a$ et $s = P^\top\mathbf{1}_d - b$ :

$$[\nabla f(P)]_{ij} = C_{ij} + \lambda r_i + \lambda s_j = C_{ij} + \lambda [A^\top(r,s)]_{ij}$$

## Question 7 — Deux méthodes de gradient pour le problème pénalisé

### Méthode 1 : Descente de gradient projetée (PGD)

$$P_{k+1} = \mathrm{proj}_{\geq 0}\bigl[P_k - \alpha \nabla f(P_k)\bigr], \quad \alpha = \frac{1}{L} = \frac{1}{2\lambda d}$$

Le pas $\alpha = 1/L$ garantit la décroissance de l'objectif à chaque itération. Taux de convergence : $f(P_k) - f(P^*) = O(1/k)$.

### Méthode 2 : Gradient de Nesterov accéléré (NAPG / FISTA)

$$t_{k+1} = \frac{1 + \sqrt{1 + 4t_k^2}}{2}, \quad Y_k = P_k + \frac{t_k - 1}{t_{k+1}}(P_k - P_{k-1}), \quad P_{k+1} = \mathrm{proj}_{\geq 0}\bigl[Y_k - \alpha \nabla f(Y_k)\bigr]$$

NAPG applique le pas de gradient non pas sur $P_k$, mais sur un point extrapolé $Y_k$ qui incorpore un terme de **momentum** (inertie) vers la direction de descente précédente. Ce momentum tend vers 1 au fil des itérations (car $t_k \sim k/2$). Taux : $O(1/k^2)$, ce qui est optimal pour les méthodes de gradient du premier ordre sur les fonctions lisses convexes (Nesterov, 1983).

In [ ]:
def ugot_objective(C, a, b, P, lam):
    r = P.sum(axis=1) - a
    s = P.sum(axis=0) - b
    return float(np.sum(C * P) + 0.5 * lam * (np.dot(r, r) + np.dot(s, s)))

def ugot_gradient(C, a, b, P, lam):
    r = P.sum(axis=1) - a
    s = P.sum(axis=0) - b
    return C + lam * apply_AT(r, s)

def pgd_ugot(C, a, b, P0, lam, alpha=None, n_iters=2000, track_every=1):
    d = C.shape[0]
    if alpha is None:
        alpha = 1.0 / (2.0 * lam * d)
    P = P0.copy().astype(float)
    history = {"iter": [], "obj": [], "row_viol": [], "col_viol": []}
    def _record(k):
        m = ot_metrics(C, a, b, P)
        history["iter"].append(k)
        history["obj"].append(ugot_objective(C, a, b, P, lam))
        history["row_viol"].append(m["row_violation"])
        history["col_viol"].append(m["col_violation"])
    _record(0)
    for k in range(1, n_iters + 1):
        P = proj_pos(P - alpha * ugot_gradient(C, a, b, P, lam))
        if k % track_every == 0 or k == n_iters:
            _record(k)
    return P, history

def napg_ugot(C, a, b, P0, lam, alpha=None, n_iters=2000, track_every=1):
    d = C.shape[0]
    if alpha is None:
        alpha = 1.0 / (2.0 * lam * d)
    P = P0.copy().astype(float)
    P_prev = P.copy()
    t = 1.0
    history = {"iter": [], "obj": [], "row_viol": [], "col_viol": []}
    def _record(k):
        m = ot_metrics(C, a, b, P)
        history["iter"].append(k)
        history["obj"].append(ugot_objective(C, a, b, P, lam))
        history["row_viol"].append(m["row_violation"])
        history["col_viol"].append(m["col_violation"])
    _record(0)
    for k in range(1, n_iters + 1):
        t_new = (1.0 + np.sqrt(1.0 + 4.0 * t**2)) / 2.0
        mom = (t - 1.0) / t_new
        Y = P + mom * (P - P_prev)   # point extrapole avec momentum
        P_new = proj_pos(Y - alpha * ugot_gradient(C, a, b, Y, lam))
        P_prev, P, t = P, P_new, t_new
        if k % track_every == 0 or k == n_iters:
            _record(k)
    return P, history

In [ ]:
LAMS = [1.0, 10.0, 100.0]
N_UGOT = 3000

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
fig.suptitle(r"Q7 — PGD vs NAPG : transport déséquilibré pour $\lambda \in \{1, 10, 100\}$",
             fontweight="bold")

print(f"{'lambda':>6s}  {'Methode':>8s}  {'cout':>8s}  {'row_viol':>10s}  {'col_viol':>10s}  {'dist P*':>10s}")
print("-" * 68)

for col, lam in enumerate(LAMS):
    Pg, hg = pgd_ugot (C4, a4, b4, np.zeros((2,2)), lam, n_iters=N_UGOT, track_every=10)
    Pn, hn = napg_ugot(C4, a4, b4, np.zeros((2,2)), lam, n_iters=N_UGOT, track_every=10)

    for P_m, h, name in [(Pg, hg, "PGD"), (Pn, hn, "NAPG")]:
        m = ot_metrics(C4, a4, b4, P_m)
        print(f"  {lam:>6.0f}  {name:>8s}  {m['cost']:>8.4f}  {m['row_violation']:>10.2e}"
              f"  {m['col_violation']:>10.2e}  {np.linalg.norm(P_m - P4_star):>10.4f}")

    f_ref = ugot_objective(C4, a4, b4, P4_star, lam)
    k_arr = np.array(hg["iter"][1:], dtype=float)
    init_gap = abs(hg["obj"][0] - f_ref) + 1e-16

    axes[0, col].semilogy(hg["iter"], np.array(hg["obj"]) - f_ref + 1e-16,
                           color="tab:blue",   lw=1.8, label="PGD")
    axes[0, col].semilogy(hn["iter"], np.array(hn["obj"]) - f_ref + 1e-16,
                           color="tab:orange", lw=1.8, ls="--", label="NAPG")
    axes[0, col].semilogy(k_arr, init_gap / k_arr,    color="gray", ls=":",  lw=1, label="O(1/k)")
    axes[0, col].semilogy(k_arr, init_gap / k_arr**2, color="gray", ls="-.", lw=1, label="O(1/k2)")
    axes[0, col].set(title=f"lam={lam} — ecart a f(P*)", xlabel="Iteration k")
    axes[0, col].legend(fontsize=8); axes[0, col].grid(True, which="both", alpha=0.3)

    axes[1, col].semilogy(hg["iter"], np.array(hg["row_viol"]) + np.array(hg["col_viol"]) + 1e-16,
                           color="tab:blue",   lw=1.8, label="PGD")
    axes[1, col].semilogy(hn["iter"], np.array(hn["row_viol"]) + np.array(hn["col_viol"]) + 1e-16,
                           color="tab:orange", lw=1.8, ls="--", label="NAPG")
    axes[1, col].set(title=f"lam={lam} — violation des marges", xlabel="Iteration k")
    axes[1, col].legend(fontsize=8); axes[1, col].grid(True, which="both", alpha=0.3)

    if col < len(LAMS) - 1:
        print("-" * 68)

plt.tight_layout()
plt.show()

### Observation (Q7)

Les courbes révèlent un phénomène intéressant : **PGD et NAPG convergent très rapidement sur ce problème jouet** (en quelques dizaines d'itérations), si bien que les deux courbes se superposent presque complètement et qu'on ne distingue pas visuellement l'avantage théorique de NAPG.

Cela s'explique par la structure du problème : en dimension $d = 2$, la matrice de coût est très petite et la constante de Lipschitz $L = 2\lambda d$ est faible, ce qui donne un pas $\alpha = 1/L$ grand relativement à la taille du problème. PGD converge donc en très peu d'itérations et NAPG n'a pas le temps de montrer son avantage $O(1/k^2)$ vs $O(1/k)$. Les taux théoriques en pointillés gris montrent bien que les deux méthodes convergent **bien plus vite** que les bornes théoriques sur ce cas particulier.

Sur la violation des marges, on observe un plateau : les contraintes ne sont jamais satisfaites exactement, quelle que soit la valeur de $\lambda$. C'est inhérent au problème pénalisé — on minimise $f$, qui n'impose pas $P\mathbf{1} = a$ exactement. La violation résiduelle est de l'ordre de $1/\lambda$ : pour $\lambda = 1$ elle reste autour de 1.35, pour $\lambda = 100$ elle descend à environ $0.1$. Pour obtenir des contraintes respectées à $10^{-3}$, il faudrait $\lambda \sim 1000$ ou plus.

On notera aussi que pour $\lambda$ plus grand, la distance à $P^*$ diminue, confirmant que $P^*_\lambda \to P^*_{\text{LP}}$ quand $\lambda \to +\infty$.

---

## Question 8 — Comparaison finale sur le transfert de couleurs

In [ ]:
def color_transfer_unbalanced(src, tgt, method, lam=50.0, n_samples=300,
                               alpha=None, n_iters=4000, seed=0):
    d = n_samples
    Xs, _ = sample_pixels(src, d, seed=seed)
    Xt, _ = sample_pixels(tgt, d, seed=seed + 1)
    a = np.ones(d)
    b = np.ones(d)
    C = np.sqrt(((Xs[:, None, :] - Xt[None, :, :]) ** 2).sum(axis=-1))
    if alpha is None:
        alpha = 1.0 / (2.0 * lam * d)
    t0 = time.time()
    track_every = max(1, n_iters // 200)
    if method == "pgd":
        P, history = pgd_ugot (C, a, b, np.zeros((d, d)), lam, alpha=alpha,
                                n_iters=n_iters, track_every=track_every)
    elif method == "napg":
        P, history = napg_ugot(C, a, b, np.zeros((d, d)), lam, alpha=alpha,
                                n_iters=n_iters, track_every=track_every)
    else:
        raise ValueError(f"Methode inconnue : {method!r}")
    runtime = time.time() - t0
    T_Xs = transport_to_colors(P, a, Xt)
    out = recolor_image(src, Xs, T_Xs)
    return {"out": out, "P": P, "metrics": ot_metrics(C, a, b, P),
            "history": history, "runtime": runtime}

In [ ]:
LAM_CT = 50.0
N_ITERS_UNBAL = 4000

print(f"lambda = {LAM_CT},  alpha = 1/(2*lambda*d) = {1/(2*LAM_CT*N_SAMPLES):.2e}")
print("Execution PGD et NAPG...")

res_pgd  = color_transfer_unbalanced(src, tgt, "pgd",  lam=LAM_CT,
                                      n_samples=N_SAMPLES, n_iters=N_ITERS_UNBAL)
res_napg = color_transfer_unbalanced(src, tgt, "napg", lam=LAM_CT,
                                      n_samples=N_SAMPLES, n_iters=N_ITERS_UNBAL)

print(f"\n{'Methode':12s}  {'cout':>8s}  {'row_viol':>10s}  {'col_viol':>10s}  {'temps':>8s}")
print("-" * 55)
for name, res in [("PGD (Q7)",   res_pgd),
                   ("NAPG (Q7)",  res_napg),
                   ("PDHG (Q5)",  results_ct["pdhg"]),
                   ("Sinkhorn",   results_ct["sinkhorn"])]:
    m = res["metrics"]
    print(f"{name:12s}  {m['cost']:>8.4f}  {m['row_violation']:>10.2e}"
          f"  {m['col_violation']:>10.2e}  {res['runtime']:>7.1f}s")

In [ ]:
fig, axes = plt.subplots(1, 6, figsize=(24, 4))
fig.suptitle("Q8 — Comparaison des 4 méthodes sur le transfert de couleurs", fontweight="bold")

axes[0].imshow(src); axes[0].set_title("Source"); axes[0].axis("off")
axes[1].imshow(tgt); axes[1].set_title("Cible"); axes[1].axis("off")
for ax, (name, res) in zip(axes[2:], [("PGD (Q7)",   res_pgd),
                                        ("NAPG (Q7)",  res_napg),
                                        ("PDHG (Q5)",  results_ct["pdhg"]),
                                        ("Sinkhorn",   results_ct["sinkhorn"])]):
    ax.imshow(res["out"]); ax.set_title(name); ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle("Q8 — Convergence PGD vs NAPG (transfert de couleurs)", fontweight="bold")

for name, res, col, ls in [("PGD",  res_pgd,  "tab:blue",   "-"),
                             ("NAPG", res_napg, "tab:orange", "--")]:
    h = res["history"]
    axes[0].plot(h["iter"], h["obj"], label=name, color=col, ls=ls, lw=1.8)
    viol = np.array(h["row_viol"]) + np.array(h["col_viol"])
    axes[1].semilogy(h["iter"], viol + 1e-16, label=name, color=col, ls=ls, lw=1.8)

axes[0].axhline(results_ct["sinkhorn"]["metrics"]["cost"], color="gray", ls=":",
                label="Sinkhorn (ref.)")
axes[0].set(title=r"Objectif déséquilibré $f(P_k)$", xlabel="Itération $k$")
axes[1].set(title=r"Violation $\|P_k\mathbf{1}-a\|+\|P_k^\top\mathbf{1}-b\|$",
             xlabel="Itération $k$")
for ax in axes:
    ax.legend(); ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### Observation (Q8)

Le tableau de résultats appelle plusieurs remarques.

**PGD et NAPG convergent quasi-identiquement** sur ce problème réel aussi, comme on l'avait déjà vu en Q7 sur le problème jouet. Cela peut sembler décevant au vu de l'avantage théorique $O(1/k^2)$ vs $O(1/k)$ de NAPG. L'explication est que le pas $\alpha = 1/(2\lambda d)$ est très petit pour $\lambda = 50$ et $d = 300$, ce qui donne $\alpha \approx 3.3 \times 10^{-5}$. Les deux méthodes convergent très vite dans les premières itérations (chute brutale visible sur le graphe), puis stagnent au niveau de la violation résiduelle imposée par la pénalisation. Sur ce palier, PGD et NAPG se comportent de la même façon — l'avantage asymptotique de NAPG jouerait dans la phase de descente initiale, qui est ici terminée en moins de 100 itérations.

**Les violations de marge de PGD/NAPG ($\sim 9 \times 10^{-2}$) sont nettement plus élevées que celles de PDHG ($\sim 1.8 \times 10^{-3}$).** C'est la limitation fondamentale de l'approche pénalisée avec $\lambda = 50$ : les contraintes ne sont imposées qu'approximativement. Pour obtenir des violations comparables à PDHG, il faudrait augmenter $\lambda$ (ce qui durcit le problème et ralentit la convergence) ou utiliser une stratégie de $\lambda$ croissant.

**Visuellement**, les quatre méthodes donnent des résultats comparables sur ces images synthétiques. Avec les vraies images du notebook (ossau.jpg et rer.jpg), les différences seraient plus visibles, notamment entre GDA (corrompu) et les autres.

**Bilan final :** Sinkhorn reste la méthode la plus rapide en pratique grâce à la régularisation entropique. PDHG est préférable quand on veut résoudre le LP exact sans régularisation. PGD et NAPG sont des alternatives simples (sans variable duale) mais nécessitent de bien calibrer $\lambda$ pour que les contraintes soient approximativement respectées.

---

## Tableau récapitulatif des méthodes

| Méthode | Problème résolu | Variable duale | Taux | Contraintes exactes |
|---------|----------------|:--------------:|------|:-------------------:|
| GDA | LP contraint | oui | diverge | non |
| Alt-GDA | LP contraint | oui | $O(0.96^{k/2})$ | oui |
| PDHG | LP contraint | oui | $O(0.96^{k/2})$ | oui |
| Sinkhorn | LP régularisé (entropique) | non | exponentiel | approx. |
| PGD | LP pénalisé (quadratique) | non | $O(1/k)$ | approx. |
| NAPG | LP pénalisé (quadratique) | non | $O(1/k^2)$ | approx. |